# Entrega - Agente Santi (Minimax + Alpha-Beta)

Este notebook documenta el estudio del agente con enfoque en la variable numerica **DEPTH** (lookahead).

Objetivos:
- medir rendimiento vs jugador aleatorio por color,
- medir autodesempeño (self-play),
- cuantificar costo computacional (tiempo por jugada),
- soportar conclusiones con graficas.

In [1]:
import sys
import time
import random
from pathlib import Path
import importlib

import numpy as np
import matplotlib.pyplot as plt

# Ubica automaticamente la carpeta tournament para que los imports funcionen
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'connect4').exists():
    ROOT = ROOT.parent

if not (ROOT / 'connect4').exists():
    raise RuntimeError('No se encontro la carpeta connect4. Ejecuta este notebook dentro del proyecto.')

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from connect4.connect_state import ConnectState

SantiPolicy = importlib.import_module('groups.Group Santi.policy').SantiPolicy
RandomPolicy = importlib.import_module('groups.Group B.policy').Hello

print('Project root:', ROOT)
print('SantiPolicy:', SantiPolicy)
print('RandomPolicy:', RandomPolicy)

Project root: D:\USabana\2026-1\IA\Proyecto-Final-Fundamentos-de-IA\tournament
SantiPolicy: <class 'groups.Group Santi.policy.SantiPolicy'>
RandomPolicy: <class 'groups.Group B.policy.Hello'>


In [2]:
def _safe_mount(policy):
    try:
        policy.mount(None)
    except TypeError:
        policy.mount()


def play_single_game(red_cls, yellow_cls, measure_santi=False):
    red = red_cls()
    yellow = yellow_cls()
    _safe_mount(red)
    _safe_mount(yellow)

    state = ConnectState()
    santi_time = 0.0
    santi_moves = 0

    while not state.is_final():
        current_policy = red if state.player == -1 else yellow
        is_santi_turn = measure_santi and isinstance(current_policy, SantiPolicy)

        if is_santi_turn:
            t0 = time.perf_counter()
            action = int(current_policy.act(state.board))
            santi_time += time.perf_counter() - t0
            santi_moves += 1
        else:
            action = int(current_policy.act(state.board))

        state = state.transition(action)

    return state.get_winner(), santi_moves, santi_time


def eval_vs_random(depth, games_per_color=60):
    SantiPolicy.DEPTH = int(depth)

    red_wins = red_draws = red_losses = 0
    yellow_wins = yellow_draws = yellow_losses = 0
    total_santi_time = 0.0
    total_santi_moves = 0

    # Santi rojo (-1)
    for _ in range(games_per_color):
        winner, m, t = play_single_game(SantiPolicy, RandomPolicy, measure_santi=True)
        total_santi_moves += m
        total_santi_time += t
        if winner == -1:
            red_wins += 1
        elif winner == 0:
            red_draws += 1
        else:
            red_losses += 1

    # Santi amarillo (1)
    for _ in range(games_per_color):
        winner, m, t = play_single_game(RandomPolicy, SantiPolicy, measure_santi=True)
        total_santi_moves += m
        total_santi_time += t
        if winner == 1:
            yellow_wins += 1
        elif winner == 0:
            yellow_draws += 1
        else:
            yellow_losses += 1

    red_total = red_wins + red_draws + red_losses
    yellow_total = yellow_wins + yellow_draws + yellow_losses

    return {
        'depth': int(depth),
        'games_per_color': int(games_per_color),
        'red_wins': red_wins,
        'red_draws': red_draws,
        'red_losses': red_losses,
        'yellow_wins': yellow_wins,
        'yellow_draws': yellow_draws,
        'yellow_losses': yellow_losses,
        'red_win_rate': red_wins / red_total if red_total else 0.0,
        'yellow_win_rate': yellow_wins / yellow_total if yellow_total else 0.0,
        'global_win_rate': (red_wins + yellow_wins) / (red_total + yellow_total),
        'avg_santi_ms_per_move': 1000.0 * total_santi_time / max(total_santi_moves, 1),
    }


def eval_self_play(depth, games=40):
    SantiPolicy.DEPTH = int(depth)

    red_wins = yellow_wins = draws = 0
    for _ in range(games):
        winner, _, _ = play_single_game(SantiPolicy, SantiPolicy, measure_santi=False)
        if winner == -1:
            red_wins += 1
        elif winner == 1:
            yellow_wins += 1
        else:
            draws += 1

    return {
        'depth': int(depth),
        'self_red_wins': red_wins,
        'self_yellow_wins': yellow_wins,
        'self_draws': draws,
        'self_draw_rate': draws / max(games, 1),
    }

## Configuracion del estudio

Se puede cambiar `DEPTHS`, `GAMES_PER_COLOR` y `SELF_GAMES` segun el tiempo disponible.

In [3]:
DEPTHS = [2, 3, 4, 5]
GAMES_PER_COLOR = 40
SELF_GAMES = 30

print('DEPTHS:', DEPTHS)
print('Games vs random per color:', GAMES_PER_COLOR)
print('Self-play games:', SELF_GAMES)

DEPTHS: [2, 3, 4, 5]
Games vs random per color: 40
Self-play games: 30


In [6]:
# Estimador rapido de tiempo (micro-benchmark).
# Sirve para decidir si incluir depth 5 o 6 antes de lanzar corridas largas.

def quick_timing(depth, probe_games_per_color=4):
    t0 = time.perf_counter()
    _ = eval_vs_random(depth, games_per_color=probe_games_per_color)
    dt = time.perf_counter() - t0
    games = 2 * probe_games_per_color
    return dt / games

timing_probe = {}
for d in DEPTHS:
    avg_sec_per_game = quick_timing(d, probe_games_per_color=4)
    timing_probe[d] = avg_sec_per_game

print('Promedio aproximado por partida (segundos):')
for d in DEPTHS:
    print(f'  depth={d}: {timing_probe[d]:.3f}s/game')

estimated_total_sec = sum(timing_probe[d] * (2 * GAMES_PER_COLOR + SELF_GAMES) for d in DEPTHS)
print(f'Estimacion total del estudio actual: {estimated_total_sec/60:.1f} min')

Promedio aproximado por partida (segundos):
  depth=2: 0.109s/game
  depth=3: 0.263s/game
  depth=4: 1.702s/game
  depth=5: 3.045s/game
Estimacion total del estudio actual: 9.4 min


In [ ]:
vs_random_results = []
self_play_results = []

for d in DEPTHS:
    print(f'Corriendo depth={d} ...')
    vs_random_results.append(eval_vs_random(d, games_per_color=GAMES_PER_COLOR))
    self_play_results.append(eval_self_play(d, games=SELF_GAMES))

print('Listo.')

for row in vs_random_results:
    print(row)

for row in self_play_results:
    print(row)

In [ ]:
# Guarda resultados crudos para incluirlos en el PDF si hace falta.

import json
out_dir = ROOT / 'groups' / 'Group Santi' / 'results'
out_dir.mkdir(parents=True, exist_ok=True)

with open(out_dir / 'vs_random_depth_sweep.json', 'w', encoding='utf-8') as f:
    json.dump(vs_random_results, f, indent=2)

with open(out_dir / 'self_play_depth_sweep.json', 'w', encoding='utf-8') as f:
    json.dump(self_play_results, f, indent=2)

print('Resultados guardados en:', out_dir)

In [ ]:
depths = [r['depth'] for r in vs_random_results]
red_wr = [r['red_win_rate'] for r in vs_random_results]
yellow_wr = [r['yellow_win_rate'] for r in vs_random_results]
global_wr = [r['global_win_rate'] for r in vs_random_results]
avg_ms = [r['avg_santi_ms_per_move'] for r in vs_random_results]
self_draw = [r['self_draw_rate'] for r in self_play_results]

plt.figure(figsize=(9, 5))
plt.plot(depths, red_wr, marker='o', label='Winrate rojo')
plt.plot(depths, yellow_wr, marker='o', label='Winrate amarillo')
plt.plot(depths, global_wr, marker='o', linestyle='--', label='Winrate global')
plt.ylim(0, 1.05)
plt.xticks(depths)
plt.xlabel('Depth')
plt.ylabel('Winrate')
plt.title('Rendimiento vs Aleatorio por Profundidad')
plt.grid(alpha=0.25)
plt.legend()
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(depths, avg_ms, marker='o', color='tab:orange')
plt.xticks(depths)
plt.xlabel('Depth')
plt.ylabel('ms por jugada de Santi')
plt.title('Costo Computacional por Profundidad')
plt.grid(alpha=0.25)
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(depths, self_draw, marker='o', color='tab:green')
plt.ylim(0, 1.05)
plt.xticks(depths)
plt.xlabel('Depth')
plt.ylabel('Draw rate en self-play')
plt.title('Autodesempeno por Profundidad')
plt.grid(alpha=0.25)
plt.show()

## Lectura rapida de resultados

Puntos recomendados para concluir en el informe:
- donde deja de subir claramente el winrate al aumentar depth,
- cuanto crece el tiempo por jugada al subir depth,
- si depth alto mejora robustez por color o solo en un color,
- si conviene una configuracion de torneo (ej. depth 4) por balance costo-beneficio.